In [7]:
!pip uninstall opencv-python opencv-contrib-python opencv-python-headless
!pip install opencv-contrib-python



In [8]:
from psd_tools import PSDImage
from PIL import Image
import cv2
import numpy as np

def load_psd(psd_path):
    psd = PSDImage.open(psd_path)
    image = psd.composite()
    return image.convert("RGBA")

def detect_objects(image):
    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGBA2GRAY)
    edges = cv2.Canny(gray, 100, 200)

    contours, _ = cv2.findContours(
        edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    objects = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        area = w * h

        if area > 1500:
            objects.append({
                "bbox": (x, y, w, h),
                "area": area
            })

    return objects

def compute_saliency(image):
    saliency = cv2.saliency.StaticSaliencySpectralResidual_create()
    _, sal_map = saliency.computeSaliency(
        cv2.cvtColor(np.array(image), cv2.COLOR_RGBA2BGR)
    )
    return sal_map

def smart_crop(image, objects, saliency_map, target_size):
    W, H = image.size
    tw, th = target_size

    # Priority center = weighted mean of objects
    cx = sum(x + w/2 for x,y,w,h in [o["bbox"] for o in objects]) / len(objects)
    cy = sum(y + h/2 for x,y,w,h in [o["bbox"] for o in objects]) / len(objects)

    left = int(max(0, cx - tw / 2))
    top  = int(max(0, cy - th / 2))

    right = min(left + tw, W)
    bottom = min(top + th, H)

    return image.crop((left, top, right, bottom))

def reposition(objects, target_size):
    tw, th = target_size
    objects = sorted(objects, key=lambda o: -o["area"])

    layout = []
    x_cursor = 20

    for obj in objects:
        _, _, w, h = obj["bbox"]
        scale = min((th * 0.8) / h, 1.0)

        layout.append({
            "bbox": obj["bbox"],
            "scale": scale,
            "position": (x_cursor, int((th - h*scale)/2))
        })

        x_cursor += int(w * scale) + 20

    return layout

from PIL import Image

def render(image, layout, target_size):
    canvas = Image.new("RGBA", target_size)

    for item in layout:
        x, y, w, h = item["bbox"]
        crop = image.crop((x, y, x+w, y+h))
        resized = crop.resize(
            (int(w * item["scale"]), int(h * item["scale"]))
        )
        canvas.paste(resized, item["position"], resized)

    return canvas

TARGET_SIZE = (970, 90)

image = load_psd("D:/Datanodes_Assignment/input/Axis_Multicap_fund.psd")
objects = detect_objects(image)
saliency = compute_saliency(image)

cropped = smart_crop(image, objects, saliency, TARGET_SIZE)
layout = reposition(objects, TARGET_SIZE)

final = render(cropped, layout, TARGET_SIZE)
final.save("D:/Datanodes_Assignment/chat_gpt/attempt3/970x90/banner.png")


AttributeError: module 'cv2' has no attribute 'saliency'